<a href="https://colab.research.google.com/github/Aeman-Imtiaz/Parallel-and-Distributed-Computing/blob/main/Threads.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%bash
cat <<EOF > code.c

#include <stdio.h>
#include <pthread.h>
#include <unistd.h>

pthread_mutex_t roomA, roomB;  // Two rooms, two locks

void* student1(void* arg) {
    pthread_mutex_lock(&roomA);
    printf("Student 1: Locked Room A\n");
    sleep(1);  // Using Room A

    pthread_mutex_lock(&roomB);  // Needs Room B → WAITS FOREVER
    printf("Student 1: Locked Room B\n");  // NEVER prints

    pthread_mutex_unlock(&roomB);
    pthread_mutex_unlock(&roomA);
    return NULL;
}

void* student2(void* arg) {
    pthread_mutex_lock(&roomB);
    printf("Student 2: Locked Room B\n");
    sleep(1);  // Using Room B

    pthread_mutex_lock(&roomA);  // Needs Room A → WAITS FOREVER
    printf("Student 2: Locked Room A\n");  // NEVER prints

    pthread_mutex_unlock(&roomA);
    pthread_mutex_unlock(&roomB);
    return NULL;
}

int main() {
    pthread_mutex_init(&roomA, NULL);
    pthread_mutex_init(&roomB, NULL);

    pthread_t s1, s2;
    pthread_create(&s1, NULL, student1, NULL);
    pthread_create(&s2, NULL, student2, NULL);

    pthread_join(s1, NULL);  // NEVER finishes
    pthread_join(s2, NULL);  // NEVER finishes

    printf("Both students finished!\n");  // NEVER prints
    return 0;
}
//
Process is interrupted.
EOF

gcc code.c -o code
./code

trylock:

In [ ]:
%%bash
cat <<EOF > code.c

#include <stdio.h>
#include <pthread.h>
#include <unistd.h>

pthread_mutex_t roomA, roomB;

void* student1(void* arg) {
    pthread_mutex_lock(&roomA);
    printf("Student 1: Locked Room A - Studying Math\n");
    sleep(3);  // Long study session

    printf("Student 1: Need Room B for reference book...\n");
    pthread_mutex_lock(&roomB);  // WAITS FOREVER - DEADLOCK!
    printf("Student 1: Got Room B! Complete!\n");

    pthread_mutex_unlock(&roomB);
    pthread_mutex_unlock(&roomA);
    return NULL;
}

void* student2(void* arg) {
    pthread_mutex_lock(&roomB);
    printf("Student 2: Locked Room B - Drawing Art\n");
    sleep(1);

    // TRYLOCK - The smart student!
    printf("Student 2: Need Room A for art supplies...\n");

    if (pthread_mutex_trylock(&roomA) == 0) {
        printf("Student 2: Got Room A! Complete!\n");
        pthread_mutex_unlock(&roomA);
    } else {
        printf("Student 2: Room A is busy! Can't wait.\n");
        printf("Student 2: Instead, I'll go to the library to read.\n");
        pthread_mutex_unlock(&roomB);
        printf("Student 2: Released Room B. Will try again after library.\n");

        // DO SOMETHING ELSE INSTEAD OF WAITING!
        printf("Student 2: Reading at library for 2 seconds...\n");
        sleep(2);

        // Try again later
        printf("Student 2: Trying again...\n");
        pthread_mutex_lock(&roomB);
        if (pthread_mutex_trylock(&roomA) == 0) {
            printf("Student 2: Got both rooms this time! Complete!\n");
            pthread_mutex_unlock(&roomA);
            pthread_mutex_unlock(&roomB);
            //Deadlock is avoided by maintaining a consistent lock acquisition order; unlocking order does not fix deadlock if threads acquire locks in opposite order.
        }
    }
    return NULL;
}

int main() {
    pthread_mutex_init(&roomA, NULL);
    pthread_mutex_init(&roomB, NULL);

    pthread_t s1, s2;
    pthread_create(&s1, NULL, student1, NULL);
    pthread_create(&s2, NULL, student2, NULL);

    pthread_join(s1, NULL);
    pthread_join(s2, NULL);

    printf("\n Both students finished!\n");
    return 0;
}

EOF

gcc code.c -o code
./code

In [ ]:
%%bash
cat <<EOF > code.c
#include <stdio.h>
#include <pthread.h>
#include <unistd.h>

pthread_mutex_t lock;

void* task(void* arg) {
    int id = *(int*)arg;

    if(pthread_mutex_trylock(&lock) == 0) {
        printf("Thread %d got lock\n", id);

        sleep(2);

        printf("Thread %d releasing lock\n", id);
        pthread_mutex_unlock(&lock);
    }
    else {
        printf("Thread %d could NOT get lock (skipping)\n", id);
    }
}

int main() {
    pthread_t t1, t2;
    int id1 = 1, id2 = 2;

    pthread_mutex_init(&lock, NULL);

    pthread_create(&t1, NULL, task, &id1);
    pthread_create(&t2, NULL, task, &id2);

    pthread_join(t1, NULL);
    pthread_join(t2, NULL);
}
EOF

gcc code.c -o code
./code